In [6]:
#import environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_openai import ChatOpenAI

model=ChatOpenAI(
    model='glm-4-flash',
    openai_api_base="https://open.bigmodel.cn/api/paas/v4/",
    max_tokens=300,
    temperature=0.7)

通过装饰器定义属于自己的tool，写一个函数加上描述它函数功能的docstring，通过docstring可以知道这个函数是干吗的，在合适的时间调用它

In [1]:
from langchain_community.tools import DuckDuckGoSearchRun

import pandas as pd
from pandasql import sqldf

def simulate_database_operation(sql):
	# 用 dataframe 假装一个表
	my_table = pd.DataFrame({
	    'name': ['Henry Myers', 'Martha Hawkins', 'Kelsey Lutz', 'Jonathan Fowler', 'Jonathan Young', 'Autumn Johnson', 'Kimberly Macias', 'Jared Mccormick', 'Casey Hoover', 'Erica Morse'],
	    'age': [60, 44, 54, 46, 76, 22, 69, 33, 23, 35],
	    'sex': ['F', 'M', 'M', 'F', 'F', 'M', 'M', 'F', 'F', 'M']
	})
	result = sqldf(sql)
	return result

print(simulate_database_operation('SELECT * FROM my_table WHERE age > 50'))
# 错误调用
# print(simulate_database_operation('SELECT * FROM my_table WHERE id > 50'))

              name  age sex
0      Henry Myers   60   F
1      Kelsey Lutz   54   M
2   Jonathan Young   76   F
3  Kimberly Macias   69   M


将上面的函数改装成一个tool

In [2]:
from langchain.tools import tool

@tool
def simulate_database_operation(sql: str):
	'''根据sql语句操作数据库'''
	my_table = pd.DataFrame({
	    'name': ['Henry Myers', 'Martha Hawkins', 'Kelsey Lutz', 'Jonathan Fowler', 'Jonathan Young', 'Autumn Johnson', 'Kimberly Macias', 'Jared Mccormick', 'Casey Hoover', 'Erica Morse'],
	    'age': [60, 44, 54, 46, 76, 22, 69, 33, 23, 35],
	    'sex': ['F', 'M', 'M', 'F', 'F', 'M', 'M', 'F', 'F', 'M']
	})
	result = sqldf(sql)
	return result

In [8]:
tools = [DuckDuckGoSearchRun(), simulate_database_operation]
model_with_tools = model.bind_tools(tools)

from pprint import pprint as pp

response = model_with_tools.invoke('印度的首都是哪里？')
pp(dict(response))

{'additional_kwargs': {'refusal': None},
 'content': '印度的首都是新德里（New '
            'Delhi）。它位于印度的北部，是印度联邦政府的行政首都。同时，印度的立法首都位于南部的海得拉巴（Hyderabad），而印度的商业和金融中心是孟买（Mumbai）。',
 'example': False,
 'id': 'run-15ff3e4a-ff72-4c71-8813-c7058bf531db-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'stop',
                       'logprobs': None,
                       'model_name': 'glm-4-flash',
                       'system_fingerprint': None,
                       'token_usage': {'completion_tokens': 52,
                                       'completion_tokens_details': None,
                                       'prompt_tokens': 275,
                                       'prompt_tokens_details': None,
                                       'total_tokens': 327}},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input_tokens': 275,
                    'output_tokens': 52,
                    'total_tokens': 327}}


In [10]:
db_response = model_with_tools.invoke('今天有什么新鲜事，搜索互联网后告诉我')
pp(dict(db_response))

{'additional_kwargs': {'refusal': None,
                       'tool_calls': [{'function': {'arguments': '{"query": '
                                                                 '"今天有什么新鲜事"}',
                                                    'name': 'duckduckgo_search'},
                                       'id': 'call_9097932054148687803',
                                       'index': 0,
                                       'type': 'function'}]},
 'content': '',
 'example': False,
 'id': 'run-ed99d47e-b514-4bc9-8e85-5927d6f0846c-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'tool_calls',
                       'logprobs': None,
                       'model_name': 'glm-4-flash',
                       'system_fingerprint': None,
                       'token_usage': {'completion_tokens': 15,
                                       'completion_tokens_details': None,
                                       'prompt_tokens': 279,
       

In [9]:
db_response = model_with_tools.invoke('帮我往数据库的my_table表中插入一条数据，name是张三，age是18，sex是male')
pp(dict(db_response))

{'additional_kwargs': {'refusal': None,
                       'tool_calls': [{'function': {'arguments': '{"sql": '
                                                                 '"INSERT INTO '
                                                                 'my_table '
                                                                 '(name, age, '
                                                                 'sex) VALUES '
                                                                 "('张三', 18, "
                                                                 '\'male\')"}',
                                                    'name': 'simulate_database_operation'},
                                       'id': 'call_9097928515094596084',
                                       'index': 0,
                                       'type': 'function'}]},
 'content': '',
 'example': False,
 'id': 'run-75d63f63-3788-4aa8-97c9-211eb54b1561-0',
 'invalid_tool_calls': [],
 'name': None,

实现整个agent，用langgraph定义流程

In [ ]:
from icecream import ic

# 定义 agent
def manual_agent(query: str, model: ChatOpenAI, tools: list[tool]):
    model_with_tools = model.bind_tools(tools)
    model_output = model_with_tools.invoke(query)
    tool_response = call_tool(model_output, tools)
    final_response = model.invoke(
        f'original query: {query} \n\n\n tool response: {tool_response}',
    )
    return final_response


def call_tool(model_output, tools):
    tools_map = {tool.name.lower(): tool for tool in tools}
    tools_response = {}
    for tool in model_output.tool_calls:
        tool_name = tool['name']
        tool_args = tool['args']
        tool_instance = tools_map[tool_name]
        tool_response = tool_instance.invoke(*tool_args.values())
        tools_response[tool_name] = tool_response
    return tools_response

manual_agent('帮我查询数据库my_table表中有多少人年龄大于60', model, tools).content

In [ ]:
from langchain.agents import create_react_agent

# reson & act
agent = create_react_agent(model, tools)

from langchain_core.messages import HumanMessage

response = agent.invoke({'messages': [HumanMessage(content="今天北京的天气怎么样")]})
response